In [1]:
#Task 4.1
#YEAH NIGGER NIGGER NIGGER ~ I LOVE NIGGER ~

#init
import sqlite3
conn = sqlite3.connect('business.db')
conn.row_factory = sqlite3.Row
cur = conn.cursor()

#schema
cur.execute("""
DROP TABLE IF EXISTS villas
""")
cur.execute("""
DROP TABLE IF EXISTS customerBookings
""")

cur.execute("""
CREATE TABLE villas(
villaID INTEGER PRIMARY KEY,
villaname TEXT,
country TEXT,
cost INTEGER
)
""")
cur.execute("""
CREATE TABLE customerBookings(
bookingID INTEGER PRIMARY KEY,
customerID INTEGER,
villaID INTEGER,
startdate TEXT,
daycount INTEGER,
FOREIGN KEY(villaID) REFERENCES villas
)
""")


In [2]:
#Task 4.2

#seed
with open('villas.txt') as file:
    for row in file:
        row = row.strip().split(',')
        cur.execute("""
        INSERT OR IGNORE INTO villas (villaID, villaname, country, cost)
        VALUES (?,?,?,?)
        """,(int(row[0]), row[1], row[2], int(row[3])))

with open('customerBookings.txt') as file:
    for row in file:
        row = row.strip().split(',')
        cur.execute("""
        INSERT OR IGNORE INTO customerBookings (bookingID, customerID, villaID, startdate, daycount)
        VALUES (?,?,?,?,?)
        """, (int(row[0]), int(row[1]), int(row[2]), row[3], int(row[4])))
conn.commit()

In [22]:
#Task 4.3

#schema for new table
cur.execute("""
DROP TABLE IF EXISTS Villa_Booking
""")
cur.execute("""
CREATE TABLE Villa_Booking(
villaID INTEGER,
date TEXT,
PRIMARY KEY (villaID, date)
FOREIGN KEY(villaID) REFERENCES villas
)
""")

#populate new table 
"""
to populate the table with villaID and dates for the stay, we need the villaID (obviously)
as well as startdate and daycount 
for each daycount, we will increment the date and insert it with the villaID
"""

#fetch villaID, startdate and daycount ordered by villaID
cur.execute("""
SELECT villaID, startdate, daycount 
FROM customerBookings
ORDER BY villaID ASC
""")
result = cur.fetchall()

#populate villa_booking with result
import datetime
for row in result:
    for n in range(row['daycount']):
        #dissect date and add days
        day = int(row['startdate'][:2]) + n
        month = datetime.datetime.strptime(row['startdate'][3:],'%b').month
        #month rollover case handling
        leftover = 0
        if day >= 28:
            leftover = day - 28
            day = 28
        d = datetime.date(2025, month, day)
        d += datetime.timedelta(days=leftover)
        #we finally have a valid date to insert
        #reconstruct the date
        output = f'{d.strftime("%d")}-{d.strftime("%b")}'
        cur.execute("""
        INSERT OR IGNORE INTO Villa_Booking (villaID,date)
        VALUES (?,?)
        """, (row['villaID'],output))
conn.commit()

In [27]:
#Task 4.4

def menu():
    print('please key in the specifics to book a villa')
    name = input("villa name: ").strip().lower()
    month = input('start month:')
    date = int(input("start date: "))
    no = int(input('number of days: '))
    #list out all days that are being booked in the 'XX-Nov' format
    booking = []
    #initialise start date as datetime.date object
    d = datetime.date(2025,datetime.datetime.strptime(month,'%b').month,date)
    #append every single day that will be spent occupying the villa
    for i in range(no):
        d += datetime.timedelta(days=i)
        booking.append(f'{d.strftime("%d")}-{d.strftime("%b")}')
    # find results for days that are going to be booked
    cur.execute("""
    SELECT Villa_Booking.date 
    FROM Villa_Booking INNER JOIN villas ON Villa_Booking.villaID = villas.VillaID
    WHERE villas.villaname LIKE ? 
    """,(name,))
    occupied = cur.fetchall()
    #we now have a list of all dates where the villa is currently occupied 
    available = []
    unavailable = []
    fulfilled = True
    for item in booking:
        #item in occupied dates - not available at customer's request
        if item in occupied:
            print(item,' is unavailable for booking')
            unavailable.append(item)
            fulfilled = False
        else:
            available.append(item)
    if not fulfilled:
        print('we are unfortunately unable to fully fulfill your request')
        print(f'{unavailable} are all unavailable')
        print(f'however, dates {available} are still available')
    else:
        print(f'{available} are all dates available as per your request')
    

menu()

please key in the specifics to book a villa
villa name: Dolphin
start month:Apr
start date: 8
number of days: 4
['08-Apr', '09-Apr', '11-Apr', '14-Apr'] are all dates available as per your request
